# 04: Fine-tune attacks A1 / A2 (PLAN.md §4, §13.5)
**A1**: benign LoRA fine-tune on 1,000 Alpaca examples. **A2**: 100 harmful instruction to compliant-response pairs
(LLM-LAT/harmful-dataset) + 900 benign. One training block for every model (configs/attacks.yaml `train`):
QLoRA nf4, r=16, alpha=32, lr 2e-4, 1 epoch, max_len 512, batch 4 x grad-accum 4, gradient checkpointing, seed 42.

Per (model, attack): train once on the model **as released** (GC/IL adapter merged, ST deltas applied, RL = BASE
weights with BASE's template; RL's guard is attached at eval only) -> merge into fp16 -> generate every split ->
**delete the adapter**. Adapters live only in local temp storage and are never pushed (PLAN.md §9); the push code
refuses anything but result `.parquet`/`.json`.

Order: dev sanity check on BASE + SDT (A1, then A2), then all 8 runs on test with `--final` (A1, then A2).
Two GPUs: one (model, attack) group per T4.

Safe for **Save & Run All (Commit)**: no prompts; each stage first pulls finished jobs from the private Hub repo,
so a re-commit after a dead session only trains and generates what is missing. Training time per run is logged and
stored in each manifest (`attack_info.train_seconds`).

Requires: `HF_TOKEN` (write), GPU T4 x2.

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
# Fine-tune data and temp locations (adapters and merged weights stay on local disk, never in results/)
import os
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.makedirs(os.environ["AFTERMARKET_ADAPTERS"], exist_ok=True)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# Materialize ft_a2 once (rebuilt from the pinned dataset, sha256-verified; cache is gitignored)
try:
    import sys; sys.path.insert(0, "src")
    from aftermarket.ft_data import load_ft
    print({n: len(load_ft(n)) for n in ["ft_a1", "ft_a2"]})
except Exception as e:
    print(f"WARNING fine-tune data check failed: {type(e).__name__}: {e}")

In [ ]:
# Helpers: per-job stats incl. training time (stats only, never completion text)
import glob, json
import pandas as pd

def ft_table(splits=None):
    recs = []
    for p in sorted(glob.glob("results/gen/*.json")):
        try:
            m = json.load(open(p))
        except Exception:
            continue
        j, ai = m["job"], m.get("attack_info", {})
        if j["attack_id"] not in ("A1", "A2") or (splits and j["split"] not in splits):
            continue
        recs.append({"run": j["run_id"], "attack": j["attack_id"], "split": j["split"], "n": m["n_prompts"],
                     "train_s": ai.get("train_seconds"), "loss0": ai.get("loss_first10"), "loss1": ai.get("loss_last10"),
                     "peak_gb": ai.get("peak_vram_gb"), "cfg": ai.get("train_config_hash"),
                     "gen_s": m["seconds"], "pushed": m.get("pushed")})
    return pd.DataFrame(recs)

def show(splits, label):
    try:
        t = ft_table(splits)
        with pd.option_context("display.width", 220, "display.max_rows", 500):
            print(t.to_string(index=False) if len(t) else f"no {label} results yet")
        if len(t) and t.cfg.nunique() > 1:
            print("WARNING: more than one training config hash:", t.cfg.unique())
    except Exception as e:
        print(f"{label} summary failed: {type(e).__name__}: {e}")

## Phase 1: dev sanity check (BASE + SDT, no `--final`)
Expect: loss falls; A2 lowers refusal on BASE; A1 keeps MMLU close to A0.

In [ ]:
# Stage 1: A1 dev
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_ft_dev.yaml --attacks A1

In [ ]:
# Stage 2: A2 dev
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_ft_dev.yaml --attacks A2

In [ ]:
show(['benign_dev', 'harmful_dev', 'mmlu_dev'], 'dev')

## Phase 2: all 8 runs on test (`--final`, leaderboard cells)

In [ ]:
# Stage 3: A1 test
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_ft_test.yaml --final --attacks A1

In [ ]:
# Stage 4: A2 test
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_ft_test.yaml --final --attacks A2

In [ ]:
# Push anything whose upload failed during the run (parquet+json only)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_ft_test.yaml --push-pending

In [ ]:
# Final summary (always runs; stats only, no completion text)
try:
    import shutil, yaml
    from pathlib import Path
    from aftermarket.generate import make_jobs, pending_jobs
    for name, cfg_path in [("dev", "configs/gen_ft_dev.yaml"), ("test", "configs/gen_ft_test.yaml")]:
        c = yaml.safe_load(open(cfg_path))
        jobs = make_jobs(c)
        left = pending_jobs(jobs, Path(c["out_dir"]))
        print(f"{name}: {len(jobs) - len(left)}/{len(jobs)} jobs done" + (f"; pending e.g. {[j.name for j in left[:6]]}" if left else ""))
    show(['benign_dev', 'harmful_dev', 'mmlu_dev'], "dev")
    show(['harmful_test', 'mmlu_test', 'xstest_test'], "test")
    t = ft_table()
    if len(t):
        per_run = t.drop_duplicates(["run", "attack"])[["run", "attack", "train_s"]]
        print("training time per run (s):"); print(per_run.to_string(index=False))
    unpushed = [p for p in sorted(Path("results/gen").glob("*.json")) if not json.load(open(p)).get("pushed")]
    print(f"unpushed results: {len(unpushed)}")
    ad = Path(os.environ["AFTERMARKET_ADAPTERS"])
    leftover = [p.name for p in ad.iterdir()] if ad.exists() else []
    print("leftover adapters:", leftover or "none")
    for p in leftover:
        shutil.rmtree(ad / p, ignore_errors=True)
except Exception as e:
    print(f"final summary failed: {type(e).__name__}: {e}")